## **¿Qué es un Quadtree?**

Es una estructura de datos jerárquica donde cada nodo interno tiene exactamente cuatro hijos. Particiona un espacio 2D subdividiéndolo recursivamente en cuatro cuadrantes hasta que cada región tenga ≤ `capacidad` puntos.

---

## ¿Cómo se comporta?

Al insertar un punto, el árbol desciende desde la raíz comparando coordenadas para decidir en cuál de los cuatro cuadrantes cabe. Si ese cuadrante ya está lleno, se subdivide y los puntos se redistribuyen. Esto se repite hasta llegar a un nodo hoja con espacio disponible.

El resultado es que **zonas densas** (muchas entregas juntas) generan más niveles de profundidad, mientras que **zonas vacías** permanecen como hojas sin subdividir, ahorrando memoria y tiempo.

---

## ¿Cómo busca?

En una búsqueda por radio, el árbol evalúa en cada nodo si la región rectangular de ese nodo **intersecta** el círculo de búsqueda:

- Si **no intersecta** : descarta toda la rama. No revisa ningún hijo.
- Si **intersecta** : desciende a los hijos y repite.
- Si llega a una **hoja** : calcula la distancia real de cada punto y retorna los que estén dentro del radio.

Esto significa que en lugar de revisar los 10.000 puntos, solo se exploran las ramas del árbol que geográficamente se solapan con la zona de interés, lo que hace la búsqueda significativamente más rápida a medida que $n$ crece.

---

## ¿Por qué es mejor en este problema?

| Operación | Fuerza bruta | Quadtree |
|-----------|-------------|----------|
| Range Search | $O(n)$ | $O(\log n + k)$ |
| Nearest Neighbor | $O(n)$ | $O(\log n)$ |

Con 10.000 puntos dispersos en una ciudad, la mayoría de las consultas solo tocan una fracción pequeña del árbol, haciendo la diferencia de rendimiento notable y medible.

## **Construccion del QuadTree**

En este caso usamos python para construir el quadtree

Hicimos un arbol que funcione para cualquier cantidad de dimensiones, aunque en este ejercicio solo usamos 2 dimensiones, pero funciona para 3 o mas dimensiones dependiendo de los puntos que se quieran almacenar, esto lo logramos usando un efoque recursivo para construir el arbol y las busquedas.

- El numero de cuadrantes es de siempre 2 a la d donde d es el numero de dimensiones por ejemplo si tenemos 2 dimensiones el numero de cuadrantes es 4 o si es 3 el numero de cuadrantes es 8(Octree)

- Cada nodo almacena el punto medio de su region y una lista de hijos, si el nodo es hoja entonces la lista de hijos es vacia y el punto medio es el punto almacenado en ese nodo.

- La subdivision se detiene cuando el nodo tiene un solo punto o todos lo puntos son iguales, evitando la recursion infinita

## **Busqueda en el QuadTree**

- Para la busqueda por radio, usa intersecta para descartar cuadrantes que no coinciden con el area de busqueda, esto se hace para evitar explorar ramas del arbol que no nos interesan. Es un generador yield, lo que no construye toda la lista de resultados en memoria, sino que va retornando los puntos a medida que los encuentra.

- Para la busqueda KNN, usa un maxheap de tamaño k con distancias negativas para mantener eficientemente lo k vecinos encontrados hasta ahora, y se va actualizando conforme se explora el arbol. Ordena los hijos por distancia al objetivo para explorar primero los mas prometedores, reduciendo por mucho el numero de ramas que se deben visitar.

In [10]:
import math
import heapq

#definimos el constructor que recibe la lista de puntos y el hijo es un parametro opcional que se inicializa como una lista vacía si no se proporciona
class NodoQuad:
  """Nodo que almacena un punto y sus hijos en el quadtree"""
  def __init__(self, punto, hijos=None):
    self.punto=punto
    self.hijos = hijos or []

class QuadTree:
    """clase que construye el quadtree a partir de una lista de puntos"""
    #el cosnstructor recibe una lista de puntos y primero determina la dimension con el primer punto y luego calcula los limites minimos y maximos de cada dimension para luego llamar a la función recursiva que construye el árbol con estos limites
    def __init__(self, puntos):
        dimension = len(puntos[0])
        limites_min = [min(p[i] for p in puntos) for i in range(dimension)]
        limites_max = [max(p[i] for p in puntos) for i in range(dimension)]
        self.raiz = self.construirquadtree(puntos, limites_min, limites_max, dimension, 0)

    """definimos el punto medio el cual recibe los limites max y min de cada dimension"""
    def punto_medio(self, limites_min, limites_max, dimension):
        #creamos una lista de punto medio aqui guardaremos los datos de esta mitad
        puntomedio= []

        #aqui iteramos sobre i dependiendo de la dimension y calculampos nuestro punto medio con los valores minimos y maximos de nuestra lista de puntos
        for i in range(dimension):
            medio = (limites_min[i] + limites_max[i]) / 2
            puntomedio.append(medio)
        return puntomedio
  
    def indice_cuadrante(self, puntos, puntomedio, dimension):
        """construye una lista de cuadrantes donde cada cuadrante es una lista de puntos que pertenecen a ese cuadrante"""
        cuadrantes = [[] for _ in range(2**dimension)]
        #iteramos sobre cada punto y dependiendo la posicion de este en cada dimension respecto al punto medio asignamos el indice del cuadrante
        for punto in puntos:
            indice = 0
            for i in range(dimension):
                #si el punto es mayor que el punto medio en esa dimension asignamos el indice del cuadrante correspondiente
                if punto[i] > puntomedio[i]:
                    indice += 2**i
            cuadrantes[indice].append(punto)

        return cuadrantes


    def construirquadtree(self, puntos, limites_min, limites_max, dimension, profundidad=0):
        """construimos recursivamente el quadtree dividiendo el espacio en cuadrantes y asignando los puntos a cada cuadrante hasta que lleguemos a una hoja del árbol"""
    #definimos la llamada recursiva si no hay puntos que buscar definimos la hoja de nuestro arbol.
        if not puntos:
          return None

        #llamada si solo hay un punto o si en el nodo todos son iguales los puntos, definimos la hoja del arbol
        if len(puntos) == 1 or all(p == puntos[0] for p in puntos):
          return NodoQuad(puntos[0])
        
        #calculamos el punto medio de los limites actuales para dividir el espacio y asignar en el respectivo cuadrante cada punto
        sub_punto_medio = self.punto_medio(limites_min, limites_max, dimension)
        puntos_cuadrantes = self.indice_cuadrante(puntos, sub_punto_medio, dimension)


        #creamos recursivamente cada hijo del nodo actual con los puntos que le corresponden
        hijos = []
        for i in range(2**dimension):
            if not puntos_cuadrantes[i]:
                hijos.append(None)
                continue
            #determinamos para cada cuadrante los nuevos limites max y min dependiendo la posicion del cuadrante 
            nuevo_min = []
            nuevo_max = []
            #iteramos sobre cada dimension para determinar los nuevos limitas max y min dependiendo de la posicion del cuadrante
            for d in range(dimension):
                superior = (i >> d) & 1
                if superior:
                    nuevo_min.append(sub_punto_medio[d])
                    nuevo_max.append(limites_max[d])
                else:
                    nuevo_min.append(limites_min[d])
                    nuevo_max.append(sub_punto_medio[d])
            #hacemos la llamada recursiva para construir el hijo del nodo actual con los puntos correspondientes y los limites nuevos   
            hijos.append(self.construirquadtree(puntos_cuadrantes[i], nuevo_min, nuevo_max, dimension, profundidad + 1))
        return NodoQuad(sub_punto_medio, hijos)
    

    """intersección entre el radio de búsqueda y el cuadrante del nodo actual, esto se hace para determinar si es necesario explorar ese nodo o no"""
    def intersecta(self, indice, punto, objetivo, radio):
        #iteramos sobre cada dimension para determinar si el cuadrante del nodo actual intersecta con el area de busqueda
        for d, coord in enumerate(objetivo):
            bit = (indice >> d) & 1
            if bit == 0 and coord - radio > punto[d]:
                return False
            if bit == 1 and coord + radio < punto[d]:
                return False
        return True


    """Búsqueda recursiva de puntos dentro de un radio: explora hijos que intersectan el área y retorna los puntos en rango."""    
    def buscar_en_radio(self, nodo, objetivo, radio):
          #si el nodo es nulo, no hay nada que buscar
          if not nodo:
              return
          #si es una hoja vemos si el punto esta dentro del radio
          if not nodo.hijos:
              if math.dist(nodo.punto, objetivo) <= radio:
                  yield nodo.punto
              return
          #si no es hoja exploramos los hijos que intersectan el area de busqueda
          for i, hijo in enumerate(nodo.hijos):
              if hijo and self.intersecta(i, nodo.punto, objetivo, radio):
                  yield from self.buscar_en_radio(hijo, objetivo, radio)


    """funcion para busquedad KNN que utilizamos una lista heap para mantener los mas cercanos y se vaya actualizando conforme exploramos el arbol"""
    def buscar_vecinos_cercanos(self, objetivo, k=1):
          mejores = []
          #funcion recursiva para explorar al arbol y actualizar la lista de mejores vecinos encontrados hasta ahora
          def recursivo(nodo):
                if not nodo: return
                #si es una hoja, calculamos la distancia al objetivo y actualizamos la lista de mejores vecinos
                if not nodo.hijos:
                    dist = -math.dist(nodo.punto, objetivo)
                    if len(mejores) < k:
                        heapq.heappush(mejores, (dist, nodo.punto))
                    else:
                        heapq.heappushpop(mejores, (dist, nodo.punto))
                    return
                #ordenamos los hijos por distancia al objetivo para explorar primero los más cercanos
                hijos_validos = [(i, h) for i, h in enumerate(nodo.hijos) if h]
                hijos_ordenados = sorted(hijos_validos, key=lambda x: math.dist(x[1].punto, objetivo))
                #exploramos los hijos ordenados y solo exploramos aquellos que podrían contener puntos más cercanos que los mejores encontrados hasta ahora
                for i, hijo in hijos_ordenados:
                    radio = -mejores[0][0] if len(mejores) == k else float('inf')
                    if self.intersecta(i, nodo.punto, objetivo, radio):
                        recursivo(hijo)
          #iniciamos la búsqueda
          recursivo(self.raiz)
          return [p for _, p in sorted(mejores, reverse=True)]